# Should Games Go on Sale? Measuring the Impact of Steam Discounts

Jasper Burkey · Senior thesis, University of Washington (advisor: Prof. Philipp Eisenhauer)

---

Four times a year Valve runs a store-wide sale, and each publisher decides whether to discount and how deep. This notebook estimates what that decision does to demand and revenue, and for which games it pays, using public data on thousands of games across 15 seasonal sales since 2023 and the machine-learning methods for causal inference surveyed in Athey and Imbens (2019).

**Contents:** 1. Data · 2. Design · 3. When the effect happens · 4. Main estimates · 5. Robustness · 6. Game size · 7. Who benefits · 8. Discount depth · 9. Is the review proxy valid? · 10. Audience segments · 11. Spillovers · 12. Policy learning · 13. A buy-or-wait model · 14. Conclusions and limitations

In [ ]:
import warnings

import pandas as pd
from IPython.display import display

from auxiliary import data, estimates, event_study, plots

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)
print(estimates.settings_note())

## 1. Data

Steam does not publish sales, so the data come from public sources, built by the scripts in [`pipeline/`](pipeline/README.md):

* **Reviews** (Steam's review API): daily review activity since January 2023, a stand-in for purchases. Exact window counts for the 645 largest games, verified to match full downloads.
* **Prices** (IsThereAnyDeal): when each game was discounted, and how deeply.
* **Sale dates:** Valve's announced seasonal sales (`pipeline/sale_calendar.csv`).
* **Game characteristics:** release dates and publishers from a daily store collector ([`collector/`](collector/README.md)).

The population is the 31,823 released, paid games with at least 50 reviews. The sample is 3,850 of them, stratified by size so that every large game is included. The unit of analysis is one game in one sale; `data/events.parquet` holds one row per game-event.

In [ ]:
events = data.load_events()
data.check_columns(events)
main_df = data.analysis_sample(events)
long_df = data.long_sample(main_df)
print(
    f"{len(events):,} game-events: {events['app_id'].nunique():,} games x {events['sale'].nunique()} sales"
)
print(
    f"Analysis sample: {len(main_df):,} game-events from {main_df['app_id'].nunique():,} games, "
    f"{main_df['discounted'].mean():.0%} discounted; {len(long_df):,} with six weeks of follow-up"
)
display(data.exclusion_table(events))
display(data.band_summary(events))

## 2. Design

Within each sale, games that discounted are compared with games that didn't, adjusting for what was known before the sale: review rate and trend, rating, age, price, a year of discount history, the response to the previous sale and the publisher's discounting habits. Comparing within the same sale removes the sale-wide surge in traffic. The identifying assumption is unconfoundedness given that pre-sale history.

**Outcomes** are log changes relative to each game's own pre-sale baseline (exp(effect) − 1 is roughly the % change): reviews, and a **revenue proxy** that values each window's reviews at the price actually paid. Since revenue is units times price, the change in log revenue is the change in log units (proxied by reviews) plus the log of the price paid relative to the regular price.

**Estimation** follows Athey and Imbens: doubly robust scores with an ensemble of LASSO, random forest and boosting, cross-fitting by game, and standard errors clustered by game. The headline, fixed in advance, is the effect for games that discounted (ATT) on the net revenue proxy.

## 3. When the effect happens

Discounted minus non-discounted games in the same sale, day by day, relative to each game's baseline. This shows the shape over time; single days have few reviews, so the size comes from section 4.

In [ ]:
curve, n_ev = event_study.event_study(events, data.load_daily())
plots.plot_event_study(curve, n_ev)
display(event_study.summarize(curve).to_frame("average difference"))

## 4. Main estimates

Four outcomes, five estimators each: the doubly robust effect for games that discounted (ATT, the headline), the average over all games (ATE), the effect for games that didn't discount (ATU), the partially linear model (PLR), and approximate residual balancing.

In [ ]:
main_table, e_main = estimates.main_estimates(main_df, long_df)
display(estimates.tidy(main_table))

**Overlap.** Discounting is highly predictable from a game's history, so many propensity scores sit near 1. The ATT needs comparable non-discounted games only where discounted games are, which is why it is the headline; the ATU relies most on extrapolation.

In [ ]:
plots.plot_overlap(e_main, main_df["discounted"].to_numpy(int))
print(
    f"Share of game-events with propensity in [0.1, 0.9]: {((e_main >= 0.1) & (e_main <= 0.9)).mean():.0%}"
)

## 5. Robustness

The log(1 + x) scale compresses changes for games with few reviews (most game-days have none), so the headline is checked on busier games and with ratio estimates, which compare mean counts directly. Winter Sales 2023 and 2024 began two to three weeks after Autumn Sales, so their baselines overlap; they are dropped as a check.

In [ ]:
display(estimates.tidy(estimates.robustness(main_df, long_df, e_main)))

## 6. Game size

Effects within each size band. For small games, use the ratio estimates (the log scale compresses their small counts); for large games, compression is negligible and the log estimates are more precise.

In [ ]:
display(estimates.tidy(estimates.by_size(long_df)))

## 7. Who benefits

An honest forest on the doubly robust scores predicts each game-event's effect; generic machine-learning inference (Chernozhukov et al.) then tests on held-out games whether the predicted differences are real (BLP), shows effects by fifth of predicted effect (GATES), and compares the most- and least-affected fifths (CLAN). Busier games are the main sample, to avoid the compression artifact; the overlap region repeats the test where discounted and non-discounted games are directly comparable.

In [ ]:
blp, gates, clan = estimates.heterogeneity(main_df)
display(blp)
plots.plot_gates(gates);

In [ ]:
keep = [
    "propensity",
    "disc_starts_365",
    "days_since_discount",
    "mean_cut_365",
    "age_days",
    "regular_usd",
    "pre_trend",
]
for (sample, outcome), t in clan.groupby(["sample", "outcome"], sort=False):
    print(f"{sample} | {outcome}: most- vs least-affected fifth")
    show = t[t["characteristic"].isin(keep)]
    display(
        show[
            [
                "characteristic",
                "most_affected",
                "least_affected",
                "difference",
                "p_adjusted",
            ]
        ].round(3)
    )

## 8. Discount depth

Each depth (1–33%, 34–60%, 61%+) versus not discounting, for all game-events (ATE) and for the game-events that chose that depth (ATT). Sale buyers often review after the sale ends, and the standard proxy values those reviews at full price; the lag-adjusted version values the two weeks after the sale at the sale price, which over-corrects, so the two bracket the truth.

In [ ]:
display(estimates.depth(main_df))

## 9. Is the review proxy valid? Steam's revenue charts

Steam publishes a weekly top-100 chart ranked by revenue. If reviews track sales, discounting should move games up that chart the way it moves reviews. For the 807 large and very large games (8,044 game-events), each week on the chart scores 101 − rank points, and the same doubly robust estimator compares discounted and non-discounted games within each sale (script `pipeline/10_rank_validation.py`; full ensemble settings).

In [ ]:
from auxiliary import saved

display(saved.rank_validation())

Discounting raises chart points by about 3.6 during the sale and raises the chance of charting by 7 percentage points, while the revenue proxy says revenue during the sale falls by about 47% (exp(−0.635) − 1). The two disagree on direction, so the proxy's revenue cost for big games is suspect.

The **weighting test** asks how Steam's revenue ranking actually weighs review growth against the price cut. The proxy assumes they count equally (revenue = units × price). Among game-events on the chart in both the baseline and sale weeks, the change in rank responds to reviews but barely to the price paid:

In [ ]:
wt, n_wt = saved.weighting_test()
print(f"Change in log rank on the two parts of the revenue proxy, {n_wt} game-events")
display(wt)

The price coefficient is 0.07 against 0.53 for reviews (difference −0.46, SE 0.10). For big games, the revenue proxy likely **overstates** what discounting costs. The policy analysis below uses this as a sensitivity check (a "chart-calibrated" revenue measure that weighs the price cut at 0.072 / 0.534 of a review change). The chart covers only the top 100, so this speaks to big games only.

## 10. Audience segments

Games are placed by **who reviews them** rather than by their tags: two games are close if the same people review both. Positions come from factorizing co-reviews (SPPMI-SVD) over 7.3 million reviews, 3.5 million after cleaning, from 1.04 million reviewers of two or more games. Games with few reviews are placed from their tags with a ridge mapping trained on the audience positions (script `pipeline/11_segments.py`).

The number of factors K was chosen by held-out prediction: for 20,000 test reviewers, hide one game they reviewed and check whether it is among the 20 games closest to the rest.

In [ ]:
k, tag_map, rob, choice = saved.segment_checks()
display(k)
print(
    f"Tag-to-audience mapping (held-out games): finds {tag_map['audience_neighbors_found_top20_predicted']:.0%} "
    f"of a game's true top-20 audience neighbors, vs {tag_map['audience_neighbors_found_top20_tags_only']:.0%} for tags alone"
)
display(rob)

K = 192 (hit rate 0.233, the smallest K within 0.01 of the best) predicts held-out reviews twice as well as tags (0.118) and five times better than popularity (0.043). Neighbors are stable when the sample is restricted to Steam purchases (74% overlap) or drops the heaviest 1% of reviewers (67%). A Gaussian mixture then groups games into 40 segments (bootstrap stability ARI 0.65). The largest segments:

In [ ]:
display(saved.segment_table(12))

Segments mix genre with language and era: Chinese-language visual novels and wuxia games form their own communities, and an "era" segment groups long-running multiplayer titles (Counter-Strike, Arma 3, PAYDAY 2, DayZ) across genres. Each game's 50 nearest neighbors define its **rivals** for the spillover test.

## 11. Spillovers: do rivals' discounts matter?

The main estimates assume a game's outcome does not depend on whether its rivals discount. Exposure is the share of a game's rivals that discounted in the sale; controls add the same share in the previous sale, so the variation used is rivals departing from their own habits. Four rival definitions are compared: the 20 nearest by audience, the 20 nearest by tags, the same tag cluster, and the same tag topic. Coefficients compare all rivals discounting with none; a typical difference between games in the same sale is about 0.13 of that (script `pipeline/12_spillovers.py`).

Four criteria were fixed before any result: a **placebo** against random rival sets, a **distance gradient** (nearer rivals should matter more), a **head-to-head** model with all definitions, and an **out-of-sample** split by period.

In [ ]:
display(saved.spillover_main())
display(saved.spillover_criteria())

There is no robust spillover. The only candidate, audience rivals for games that stayed at full price ("holdouts"), is positive rather than negative: rivals discounting slightly *raises* a holdout's reviews. It is borderline against placebos (p = 0.075), fails the gradient (the nearest five rivals matter least) and is not confirmed in 2025–26 on its own. The cluster definitions barely vary within a sale, so they have no power.

What the result does rule out is useful: the holdouts' interval for audience rivals is about [0.00, +0.35], so there is **no evidence that full-price games lose reviews when their rivals discount**, and no case for discounting defensively. For discounted games, rivals discounting too changes the gain by −0.03 (SE 0.05).

## 12. Policy learning: which games should discount, and how deep?

Doubly robust scores for each depth (none, 1–33%, 34–60%, 61%+) estimate what each game-event would get under each choice. Policy trees with one split or two levels choose a depth from eight features known before the sale: reviews per day, age, price, rating, and discount habits. Following Athey and Wager (2021) and Zhou, Athey and Wager (2023), each tree is learned on four-fifths of the games and valued on the held-out fifth, against the status quo (what publishers actually chose). Script: `pipeline/13_policy.py`.

Two design choices were fixed before results:

* **Overlap:** a policy may assign only a depth that similar games sometimes choose (propensity ≥ 0.05); otherwise the game keeps its choice. This binds for 45–70% of game-events, because most games almost always discount at about their habitual depth.
* **Robustness:** a rule counts only if it beats the status quo under both the net revenue proxy and its lag-adjusted version. The chart-calibrated measure from section 9 is a sensitivity check.

Gains over the status quo (log points, so +0.05 is about +5%), all game-events, every rule evaluated under every measure:

In [ ]:
display(saved.policy_table("all game-events"))
print(saved.policy_tree("all game-events", "Revenue proxy, net"))

One rule passes: the tree learned on the net revenue proxy beats the status quo under all three measures (+0.048, +0.038 and +0.053; SEs 0.015–0.020), with about 6% more reviews. In words: **discount every sale; shallowly (1–33%) if the game discounted within the last two to four weeks, deeply (61%+) otherwise.** It changes 14% of choices, mostly moving 34–60% discounts to 61%+. It never recommends skipping a sale.

Fixed rules are not robust: never discounting is +0.032 on the lag-adjusted measure but −0.064 chart-calibrated, so any one-size-fits-all advice depends on how revenue is measured. For busier games alone, no rule passes. The passing rule was identified among 18 candidates, so its value is somewhat optimistic; the honest test is the Winter Sale 2026, for which the rule was recorded in advance (`preregistration/winter_sale_2026.md`).

## 13. A buy-or-wait model of discounting

The causal estimates say what a discount did; a small dynamic model says why, and what other discount schedules would do. It combines three QuantEcon lecture models (script `pipeline/15_buy_or_wait.py`):

1. **Discount beliefs** (*Finite Markov Chains*): each week a game is at full price or in one of three depth groups; transitions depend on the weeks until the next Valve sale and are estimated from the price histories of the 1,927 busier games.
2. **The buyer's problem** (*Job Search*, solved with `DiscreteDP` from *Discrete State Dynamic Programming*): a potential buyer who values the game at v buys at this week's price or waits, given those beliefs and a weekly chance δ of losing interest. A share π knows the game's sale price a week ahead.
3. **Reviews and calibration:** reviews follow purchases with a lag (weekly carry-over φ). The parameters are chosen so a simulated discount reproduces the doubly robust effects for busier games in each window around the sale and by depth. Weekly patience β is set by the best fit on a grid.

The model is calibrated, not estimated: it interprets the causal results and has no standard errors.

**Fit.** Targeted moments and two held-out checks (the revenue proxy, which the model computes the same way the data do):

In [ ]:
display(saved.bw_fit())
display(saved.bw_parameters())

The fit is reasonable but not tight (sum of squared t-statistics 24.8 over 8 moments): the model understates the sale-week effect of 1–33% and 61%+ discounts and overstates the middle depth. The fit also depends on patience: at β = 0.995 the model has patient buyers who rarely lose interest (δ = 0.08); at the other values on the grid it fits worse (loss 34–41) and buyers drop out within a week or two.

**Is waiting needed?** Refitting with no waiting (buyers who don't buy at once leave, and no one knows the sale price ahead) fits ten times worse:

In [ ]:
display(saved.bw_waiting())

Without waiting the model cannot produce the dip in the week before the sale, and it badly understates the sale itself: much of the sale-week jump comes from buyers who held off for it. The elevated reviews in the weeks after the sale are review lag (a mean lag of about 2.5 weeks), not lasting extra demand.

In [ ]:
from IPython.display import Image

Image(filename=str(saved.bw_figure()))

**The revenue proxy against true revenue.** Inside the model both are known, so the proxy's bias can be measured. Effects of discounting in one sale, log points:

In [ ]:
display(saved.bw_proxy())

Two lessons. Over the six-week window, the proxy understates the revenue cost (−0.048 against a true −0.124), because sale buyers who review after the sale are valued at the full price. But over a whole year the cost is much smaller (−0.017): if a game does not discount now, many of its waiting buyers buy in the next sale at a discount anyway. A six-week window overstates what one sale's discount costs over the year.

**Discount schedules over a year.** If a publisher commits to a schedule and buyers come to expect it, revenue per year relative to never discounting:

In [ ]:
display(saved.bw_schedules(1.0))
display(saved.bw_sensitivity())

In this model every schedule earns less than never discounting, and deeper, more frequent discounts earn least (61%+ every sale: −17% at the best-fitting patience; −6% to −7% at the others). Discounts sell more copies (+2% to +12%) but at lower prices, and buyers learn to wait for them. The ordering holds when sale weeks bring two or three times the usual shoppers.

This is a result about **pure pricing**: in the model, a discount only changes the price. It leaves out what discounts may do beyond price: front-page placement and sale visibility for discounted games, new players who buy DLC or bring friends, and reviews that attract later buyers. The causal estimates include those channels; the model does not. Read together, the model says price discrimination alone does not justify frequent deep discounts, so any case for them rests on those other channels. That gap is a natural question for the player-count panel.

## 14. Conclusions and limitations

With the thesis settings, the estimates support ten conclusions (the tables above reproduce them; with quick settings the numbers differ slightly):

1. Discounting raises review activity during the sale substantially for games that discount.
2. On the revenue proxy it costs revenue during the sale, and a modest amount net of the surrounding weeks.
3. Effects are similar across game sizes once log-scale compression is accounted for.
4. Games that discount rarely gain the most.
5. Revenue effects depend on the game: discounting pays best for younger, busier, better-rated games that discount rarely and lightly, and worst for older games that discount often and deeply.
6. For the games that use them, deep discounts bring no more activity than shallow ones but cost more revenue; shallow discounts roughly break even.
7. Steam's revenue charts confirm that discounting raises sales for big games, and suggest the revenue proxy overstates its cost.
8. Rivals' discounts have no detectable effect: there is no case for discounting defensively.
9. Within the choices comparable games actually make, the best rule found is to discount every sale, shallowly after a recent discount and deeply otherwise; it gains about 4–5% over the status quo, to be confirmed on the Winter Sale 2026.
10. A calibrated buy-or-wait model needs buyers who wait for sales to fit the effects, and attributes the elevated reviews after a sale to review lag. As pure pricing, frequent deep discounts lose revenue over a year; their case rests on visibility and follow-on spending, which the model leaves out.

**Limitations.** Revenue is a proxy (reviews as units; DLC and in-game spending excluded). Identification assumes unconfoundedness given pre-sale history, and overlap is thin for games that rarely discount. Review lag is bracketed, not resolved, and follow-up ends six weeks after each sale. Developers may time updates to coincide with sales; news and patch-note controls are next.

**Next steps:** controls for updates and announcements timed with sales (news and patch notes), the out-of-sample test on the Winter Sale 2026 with the collector's player counts, matrix completion as a second estimator, and a regression discontinuity from Steam's 30-day release rule.